In [207]:
import os
from urllib.parse import urljoin
import pystac
import datetime

from pystac import Summaries
from pystac.utils import str_to_datetime

from pystac.extensions.datacube import DatacubeExtension, DimensionType, HorizontalSpatialDimensionAxis
from pystac.extensions.projection import ProjectionExtension
from pystac.extensions.scientific import ScientificExtension
from pystac.extensions.raster import RasterExtension

from shapely.geometry import mapping, box

import requests
from requests.auth import HTTPBasicAuth

from dotenv import load_dotenv
load_dotenv("/home/simon/work/stac/s5p/.env")

True

In [208]:
username = os.getenv("username")
password = os.getenv("password")

In [209]:
datacube = {
            "x": {"axis": HorizontalSpatialDimensionAxis.X,
                  "type": DimensionType.SPATIAL,
                  "extent": [4500000, 5400000],},

            "y": {"axis": HorizontalSpatialDimensionAxis.Y,
                  "type": DimensionType.SPATIAL,
                  "extent": [1800000, 1200000],},

            "time": {"type": DimensionType.TEMPORAL, 
                     "extent": ["2018-04-01T00:00:00Z", None]},
            }

In [210]:
summaries_dict = {}
summaries_dict["Doi"] = ["https://doi.org/10.5270/S5P-bj3nry0"]
summaries_dict["Epsg"] = ["27704"]
summaries_dict["Projection"] = ["Equi7Grid (EPSG: 27704)"]
summaries_dict["Timezone"] = ["UTC"]
summaries_dict["Grid"] = ["10x10km"]
summaries_dict["Temporal Resolution"] = ["daily"]

In [211]:
bbox = [9.4799695167, 46.4318173285, 16.9796667823, 49.0390742051]
properties = {"chunks": {"x":90, "y":60, "time": 30},
              "proj:code": "EPSG27704",
              "proj:bbox": [4500000, 1200000, 5400000, 1800000],
              "proj:wkt2": """PROJCS["Azimuthal_Equidistant",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.017453292519943295]],PROJECTION["Azimuthal_Equidistant"],PARAMETER["false_easting",5837287.81977],PARAMETER["false_northing",2121415.69617],PARAMETER["central_meridian",24.0],PARAMETER["latitude_of_origin",53.0],UNIT["Meter",1.0]]""",
              "proj:shape": [90, 60],
              "proj:geometry": mapping(box(*bbox)),
              "proj:transform": [10000.0, 0.0, 4500000.0, 0.0, -10000.0, 1800000.0, 0.0, 0.0, 1.0],
              #"raster:spatial_resolution": 10000,
              "cube:dimensions": datacube,
              "sci:doi": "10.5270/S5P-bj3nry0"}



In [212]:
collection = pystac.Collection(
        id="Sentinel-5P",
        title= "Sentinel-5P daily (10km)",
        description="The main objective of the Copernicus Sentinel-5P mission is to perform atmospheric measurements with high spatio-temporal resolution, to be used for air quality, ozone & UV radiation, and climate monitoring & forecasting. This STAC Collection provides all Sentinel-5P products in ZARR format. The data were gridded and reprojected to the EQUI7Grid and resampled to a pixel size of 10 km. Bilinear interpolation was used.",
        
        extent=pystac.Extent(pystac.SpatialExtent([bbox]),
                             pystac.TemporalExtent([[str_to_datetime("2018-04-01T00:00:00Z"), None]]),),

        keywords=["ESA", "Sentinel-5P", "copernicus", "Aerosol Index", "Carbon Monoxide", "Cloud", "Formaldehyde", "Methane", "Nitrogen Dioxide", "Ozone", "Sulfur Dioxide"],
        license="CC-BY-4.0",
        
        extra_fields={"cube:dimensions": datacube,},
                      #"cube:variables": parameters},
    )



In [213]:
collection.providers = [
        pystac.Provider(
            name="EODC",
            roles=[pystac.ProviderRole.HOST,
                   pystac.ProviderRole.PROCESSOR,],
            url="https://eodc.eu/",),

        pystac.Provider(
            name="ESA",
            roles=[pystac.ProviderRole.PRODUCER,
                   pystac.ProviderRole.LICENSOR,
                   pystac.ProviderRole.PROCESSOR,],
            url="https://www.esa.int/",),        
    ]

In [214]:
collection.stac_extensions = [
    ProjectionExtension.get_schema_uri(),
    DatacubeExtension.get_schema_uri(),
    ScientificExtension.get_schema_uri(),
    RasterExtension.get_schema_uri()
    ]



In [215]:
collection.summaries = Summaries(summaries_dict)
collection.extra_fields.update(properties)

In [ ]:
collection.add_asset(
    key="thumbnail",
    asset=pystac.Asset(
            href="https://objects.eodc.eu/88346baf22914e828ad2c1763e5e01ff:s5p-thumbnail/s5p_overview.png",
            media_type=pystac.MediaType.PNG,
            roles=["thumbnail"],
            title="Thumbnail")
    )

In [217]:
collection.add_link(
    pystac.Link(
        rel="store",
        target="https://data.eodc.eu/collections/S5P/S5P.zarr",
        media_type="application/octet-stream",
        title="Sentinel-5P Zarr Store",
    )
)


In [218]:
############ CO ###############

parameters = {
              "carbonmonoxide_total_column": {"description": "Vertically integrated CO column",
                     "unit": "mol m-2",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},

              "carbonmonoxide_total_column_precision": {"description": "Standard error of the vertically integrated CO column",
                     "unit": "mol m-2",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},

              "carbonmonoxide_total_column_corrected": {"description": "carbonmonoxide_total_column - carbonmonoxide_total_column_stripe_offset",
                       "unit": "1",
                       "dimensions": ["x", "y", "time"],
                       "type": "data"},
              }

from pystac.extensions.raster import RasterBand, RasterExtension
import pystac

bands = []

for var_name in parameters:
    if var_name == "qa_value":
        nodata = -99
        scale = 0.1
        dtype = "int8"
    else:
        nodata = -9999
        scale = 1e-6
        dtype = "int16"

    band = RasterBand.create(
        data_type=dtype,
        unit=parameters[var_name].get("unit"),
        nodata=nodata,
        scale=scale,
        offset=0.0,
    )
    bands.append(band)



so2_asset = pystac.Asset(
    href="https://data.eodc.eu/collections/S5P/S5P.zarr/CO",
    media_type="application/vnd.zarr; version=3",
    title="CO",
    roles=["data"],
    extra_fields={
        "cube:variables": parameters,
        "cube:dimensions": datacube,
    },
)

RasterExtension.ext(so2_asset).bands = bands

collection.add_asset("CO", so2_asset)

for band_dict, (var_name, var_meta) in zip(
    so2_asset.extra_fields["raster:bands"],
    parameters.items(),
):
    band_dict["name"] = var_name
    #band_dict["description"] = var_meta.get("description")



In [219]:
########### so2 ##############

parameters = {
              "sulfurdioxide_total_vertical_column": {"description": "total vertical column of sulfur dioxide for the polluted scenario derived from the total slant column",
                     "unit": "mol m-2",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},

              "sulfurdioxide_total_vertical_column_precision": {"description": "precision of the total vertical column of sulfur dioxide for the polluted scenario derived from the total slant column",
                     "unit": "mol m-2",
                     "dimensions": ["x", "y", "time"],                    
                     "type": "data"},
              }
from pystac.extensions.raster import RasterBand, RasterExtension
import pystac

bands = []

for var_name in parameters:
    if var_name == "qa_value":
        nodata = -99
        scale = 0.1
        dtype = "int8"
    else:
        nodata = -9999
        scale = 1e-6
        dtype = "int16"

    band = RasterBand.create(
        data_type=dtype,
        unit=parameters[var_name].get("unit"),
        nodata=nodata,
        scale=scale,
        offset=0.0,
    )
    bands.append(band)



so2_asset = pystac.Asset(
    href="https://data.eodc.eu/collections/S5P/S5P.zarr/SO2",
    media_type="application/vnd.zarr; version=3",
    title="SO2",
    roles=["data"],
    extra_fields={
        "cube:variables": parameters,
        "cube:dimensions": datacube,
    },
)

RasterExtension.ext(so2_asset).bands = bands

collection.add_asset("SO2", so2_asset)

for band_dict, (var_name, var_meta) in zip(
    so2_asset.extra_fields["raster:bands"],
    parameters.items(),
):
    band_dict["name"] = var_name
    #band_dict["description"] = var_meta.get("description")


In [220]:
############ AER_AI ###############

parameters = {
              "aerosol_index_354_388": {"description": "Aerosol index from 354 and 388 nm",
                     "unit": "1",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},

              "aerosol_index_340_380": {"description": "Aerosol index from 340 and 380 nm",
                     "unit": "1",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},

              "aerosol_index_335_367": {"description": "Aerosol index from 335 and 367 nm",
                       "unit": "1",
                       "dimensions": ["x", "y", "time"],
                       "type": "data"},
              "aerosol_index_354_388_precision": {"description": "Precision of aerosol index from 354 and 388 nm",
                       "unit": "1",
                       "dimensions": ["x", "y", "time"],
                       "type": "data"},
              "aerosol_index_340_380_precision": {"description": "Precision of aerosol index from 340 and 380 nm",
                     "unit": "1",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},

              "aerosol_index_335_367_precision": {"description": "Precision of aerosol index from 335 and 367 nm",
                       "unit": "1",
                       "dimensions": ["x", "y", "time"],
                       "type": "data"},
              }

bands = []

for var_name in parameters:
    if var_name == "qa_value":
        nodata = -99
        scale = 0.1
        dtype = "int8"
    else:
        nodata = -9999
        scale = 1e-6
        dtype = "int16"

    band = RasterBand.create(
        data_type=dtype,
        unit=parameters[var_name].get("unit"),
        nodata=nodata,
        scale=scale,
        offset=0.0,
    )
    bands.append(band)



so2_asset = pystac.Asset(
    href="https://data.eodc.eu/collections/S5P/S5P.zarr/AER_AI",
    media_type="application/vnd.zarr; version=3",
    title="AER_AI",
    roles=["data"],
    extra_fields={
        "cube:variables": parameters,
        "cube:dimensions": datacube,
    },
)

RasterExtension.ext(so2_asset).bands = bands

collection.add_asset("AER_AI", so2_asset)

for band_dict, (var_name, var_meta) in zip(
    so2_asset.extra_fields["raster:bands"],
    parameters.items(),
):
    band_dict["name"] = var_name
    #band_dict["description"] = var_meta.get("description")


In [221]:
############ HCHO ###############

parameters = {
              "formaldehyde_tropospheric_vertical_column": {"description": "vertical column of formaldehyde",
                     "unit": "mol m-2",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},

              "formaldehyde_tropospheric_vertical_column_precision": {"description": "random error of vertical column density",
                     "unit": "mol m-2",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},
              }

bands = []

for var_name in parameters:
    if var_name == "qa_value":
        nodata = -99
        scale = 0.1
        dtype = "int8"
    else:
        nodata = -9999
        scale = 1e-6
        dtype = "int16"

    band = RasterBand.create(
        data_type=dtype,
        unit=parameters[var_name].get("unit"),
        nodata=nodata,
        scale=scale,
        offset=0.0,
    )
    bands.append(band)



so2_asset = pystac.Asset(
    href="https://data.eodc.eu/collections/S5P/S5P.zarr/HCHO",
    media_type="application/vnd.zarr; version=3",
    title="HCHO",
    roles=["data"],
    extra_fields={
        "cube:variables": parameters,
        "cube:dimensions": datacube,
    },
)

RasterExtension.ext(so2_asset).bands = bands

collection.add_asset("HCHO", so2_asset)

for band_dict, (var_name, var_meta) in zip(
    so2_asset.extra_fields["raster:bands"],
    parameters.items(),
):
    band_dict["name"] = var_name
    #band_dict["description"] = var_meta.get("description")


In [222]:
############ NO2 ###############

parameters = {
              "nitrogendioxide_tropospheric_column": {"description": "Tropospheric vertical column of nitrogen dioxide",
                     "unit": "mol m-2",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},

              "nitrogendioxide_tropospheric_column_precision": {"description": "Precision of the tropospheric vertical column of nitrogen dioxide",
                     "unit": "mol m-2",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},
              "air_mass_factor_troposphere": {"description": "Tropospheric air mass factor",
                     "unit": "1",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"},
              "air_mass_factor_total": {"description": "Total air mass factor",
                     "unit": "1",
                     "dimensions": ["x", "y", "time"],
                     "type": "data"}, 
              }
bands = []

for var_name in parameters:
    if var_name == "qa_value":
        nodata = -99
        scale = 0.1
        dtype = "int8"
    elif var_name == "tm5_tropopause_layer_index":
        nodata = -99
        dtype = "int8"       
    else:
        nodata = -9999
        scale = 1e-6
        dtype = "int16"

    band = RasterBand.create(
        data_type=dtype,
        unit=parameters[var_name].get("unit"),
        nodata=nodata,
        scale=scale,
        offset=0.0,
    )
    bands.append(band)



so2_asset = pystac.Asset(
    href="https://data.eodc.eu/collections/S5P/S5P.zarr/NO2",
    media_type="application/vnd.zarr; version=3",
    title="NO2",
    roles=["data"],
    extra_fields={
        "cube:variables": parameters,
        "cube:dimensions": datacube,
    },
)

RasterExtension.ext(so2_asset).bands = bands

collection.add_asset("NO2", so2_asset)

for band_dict, (var_name, var_meta) in zip(
    so2_asset.extra_fields["raster:bands"],
    parameters.items(),
):
    band_dict["name"] = var_name
    #band_dict["description"] = var_meta.get("description")


In [223]:
collection_path = f"{collection.id}.json"
collection.set_self_href(collection_path)
collection.save_object()

In [224]:
r = requests.put("https://dev.stac.eodc.eu/ingestion/v1/collections/Sentinel-5P", json=collection.to_dict(), auth=HTTPBasicAuth(username, password), timeout = 30)

In [225]:
#r = requests.delete("https://dev.stac.eodc.eu/ingestion/v1/collections/Sentinel-5P",auth=HTTPBasicAuth(username, password), timeout = 30)

In [226]:
r

<Response [200]>